# JN4 · When is a building actually *done*?

**Curriculum notebook 4 of 6.** A building doesn't become finished all at once — it passes milestones. A permit is *issued* (the city says "you may build"), and later the building is *finaled* (the city says "people may live here"). So when we claim a home is **completed**, what exactly are we trusting?

The tempting answer is to find a status column that already says `'Completed'` and believe it. This notebook does the opposite: it turns the structured date columns into **dated events**, then **derives** each building's stage from those dates — never from a status string. That single discipline is why our completion count can point at a real date for every building it claims.

> Clonable + **read-only** — it reads the shared modules, never writes.

### Running the cells

To run a cell, click it and press **Shift + Return**, or click the **run (▸) button** on the cell. The simplest way through any notebook here is to start at the top and run each cell in order, reading the output that appears beneath it.

Some of the computational cells may look complex right now — that's expected, and it's fine. **You don't need to understand every line yet;** the ideas become clear as you go. Run them, watch what they produce, and keep moving.

💡 Tip: the **Next** link opens the following notebook in a new tab. If Colab says you have too many sessions, just close the previous tab and continue.

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← Previous: [JN3 · Build the spine + units](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN3_spine_units.ipynb)  |  Next: [JN5 · Year & cycle tagging](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN5_year_cycle.ipynb) →

## (run first) Colab setup

Fetches the data + shared modules from R2. **No-op if you already have the repo locally** (it detects a checkout and skips). On Colab / a bare session it recreates the minimal repo layout under the working directory so the config cell below finds everything unchanged.

In [ ]:
# === COLAB BOOTSTRAP - fetch curriculum data + modules from R2 (NO-OP if the repo is local) ===
from pathlib import Path
import sys, urllib.request, urllib.parse, tarfile, subprocess

R2 = 'https://pub-2cee87f70da64080ab70ee0a34b55099.r2.dev/curriculum'
USE_CLEAN = False   # False: raw .xlsx path (JN1's messy-data lesson).  True (skip-ingest): permits_clean.*

_here = Path.cwd()
def _repo_ok(_here):
    """True only if a scripts/ tree exists AND housing_rules actually imports from it.
    A stale Colab extraction satisfies 'the directory exists' while being unusable, which
    previously skipped both the module refetch AND the data fetch. Anything that cannot
    import is treated as absent; under /content (a disposable Colab tree, never a real
    checkout) the broken copy is removed so the fetch below replaces it."""
    import importlib, shutil
    for _base in [_here] + list(_here.parents):
        if not (_base/'scripts'/'build_v2').exists():
            continue
        sys.path.insert(0, str(_base/'scripts'))
        try:
            for _m in [k for k in list(sys.modules)
                       if k.split('.')[0] in ('housing_rules', 's0_keys', 'cpra_dedup')]:
                del sys.modules[_m]
            importlib.invalidate_caches()
            import housing_rules  # noqa: F401  - the real test: does the package satisfy its own __init__?
            return True
        except Exception as _e:
            print(f'modules present but unusable ({type(_e).__name__}: {_e}); refetching')
            try: sys.path.remove(str(_base/'scripts'))
            except ValueError: pass
            # Remove the broken tree ONLY where it is a downloaded extraction, never a real
            # checkout: a genuine repo has .git beside scripts/. Without this removal the
            # fetch below is skipped (its own guard also only tests existence) and the stale
            # copy survives — which is precisely the bug this replaces.
            if not (_base/'.git').exists():
                shutil.rmtree(_base/'scripts', ignore_errors=True)
                print('removed the unusable scripts/ tree; it will be re-downloaded')
            return False
    return False

_have_repo = _repo_ok(_here)

def _get(url):
    # r2.dev sits behind Cloudflare, which 403s the default 'Python-urllib' User-Agent; send a browser UA.
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

if _have_repo:
    print('local repo detected - no fetch needed')
else:
    try:
        import pyarrow  # the parquet / USE_CLEAN path needs it; Colab has pandas, maybe not pyarrow
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)
    def _fetch(url, dest):
        dest = Path(dest)
        if dest.exists():
            return                                   # cached: re-runs don't re-download
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(_get(url)); print('fetched', dest.name)
    # 1) shared modules -> ./scripts/...  (the config-cell repo-root walk then finds scripts/build_v2)
    if not (_here/'scripts'/'build_v2').exists():
        Path('modules.tgz').write_bytes(_get(f'{R2}/curriculum_modules.tar.gz'))
        _tar = tarfile.open('modules.tgz')
        try: _tar.extractall(_here, filter='data')      # py3.12+: safe extract, no deprecation warning
        except TypeError: _tar.extractall(_here)         # older python has no filter arg
        _tar.close(); Path('modules.tgz').unlink(missing_ok=True)   # tidy: drop the intermediate tarball
        print('extracted modules -> ./scripts/')
    # 2) data -> the SAME relative paths the notebooks use (raw .xlsx AND clean exports, both fetched)
    for rel in ['data/raw/cpra-downloads/BP_Annual Permit Report-2018-2022.xlsx',
                'data/raw/cpra-downloads/BP_Annual Permit Report-2023-2025.xlsx',
                'databases/hcd_apr_mirror_2026-06-17_fresh.db',
                'databases/hcd_apr_mirror.db',
                'data/processed/permits_clean.csv',
                'data/processed/permits_clean.parquet',
                'data/processed/permits_clean_README.md']:
        _fetch(f"{R2}/data/{urllib.parse.quote(rel.split('/')[-1])}", _here/rel)   # quote -> %20 for the spaced .xlsx names
    print('curriculum bundle ready (fetched from R2)')


In [ ]:
def md(t):
    from IPython.display import Markdown, display
    display(Markdown(t))

## Config

In [ ]:
# === CONFIG - point this at YOUR city's permit data (this notebook is clonable) ===
from pathlib import Path
import sys, glob
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts' / 'build_v2').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
PERMIT_GLOB = str(REPO_ROOT / 'data/raw/cpra-downloads/BP_Annual Permit Report-*.xlsx')
HEADER_ROW  = 7
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'build_v2'))
print('repo root:', REPO_ROOT)


## First, rebuild the spine we'll reason about

Where do "buildings" even come from? Earlier notebooks did the hard work: JN2's address key decides when two written addresses mean the same place, and JN3's predicates decide which permits actually *create housing*. To ask date questions here, we first need that same set of housing buildings back in memory.

**Our plan:** re-run the JN1–JN3 chain — load the permit feed, keep only housing-creating rows — so the rest of this notebook has a clean spine of buildings to attach milestones to.

In [ ]:
import pandas as pd
from collections import defaultdict
from housing_predicates import is_housing, net_units   # JN3's predicates
from s0_keys import normalize_address                   # JN2's key
from cpra_dedup import extract_master_permit            # a '-REV'/'-DEF' is a revision, not a new permit

def load(path):
    d = pd.read_excel(path, dtype=str, header=HEADER_ROW); d.columns = [str(c).strip() for c in d.columns]; return d
df = pd.concat([load(f) for f in glob.glob(PERMIT_GLOB)], ignore_index=True)
df = df[df['PermitNumber'].notna()].copy()
df = df.rename(columns={'Issuance Date': 'IssuanceDate', 'Finaled Date': 'FinaledDate'})
df['isnew'] = df['Work Type'].astype(str).str.strip() == 'New'
df = df[[is_housing(o, u, n, a) for o, u, n, a in zip(df['OccType'], df['UnitsAdded'], df['NumberUnits'], df['ADU'])]]
print(f'{len(df):,} housing permit rows')


In [ ]:
md(f'''## What just happened

We replayed the whole upstream chain in one cell and landed on **{len(df):,}** housing permit rows — every row that JN3's `is_housing` predicate accepted as actually creating homes. That's the raw material for this notebook: not buildings yet, just the permits we're allowed to trust. The next steps fold these rows up into buildings and hang dated milestones on each one.

Notice we imported the *real* `is_housing`, `net_units`, `normalize_address`, and `extract_master_permit` — the same functions the production pipeline runs — rather than re-typing teaching copies that could drift.''')

## A small trap first: do all the dates even *look* the same?

Before we trust a date column, here's a fair question: are the dates in this feed all written the same way? They are not. The city stores **Issuance** dates as `MM/DD/YYYY` and **Finaled** dates as ISO datetimes (`2022-01-14 00:00:00`) — in the *same* file. The lazy fix, `str[:10]`, happens to work on one format and silently mangles the other, quietly dropping a whole class of dates (on the first pass it cost this notebook its *permitted* buildings).

**Our plan:** stop hand-slicing strings and let `pd.to_datetime` understand both shapes — then prove it reads each format correctly.

In [ ]:
# A SMALL TRAP first: the feed mixes date formats - Issuance is 'MM/DD/YYYY', Finaled is ISO
# datetime. A naive str-slice parser silently drops one of them. Parse robustly.
def pdate(x):
    d = pd.to_datetime(str(x), errors='coerce')
    return d.strftime('%Y-%m-%d') if pd.notna(d) else None
print('issuance  09/10/2020 ->', pdate('09/10/2020'))      # MM/DD/YYYY
print('finaled   2022-01-14 00:00:00 ->', pdate('2022-01-14 00:00:00'))  # ISO datetime


In [ ]:
_us  = pdate('09/10/2020')
_iso = pdate('2022-01-14 00:00:00')
md(f'''## What just happened

`pdate` read **both** shapes into the same clean `YYYY-MM-DD` string: the `MM/DD/YYYY` issuance date became **{_us}**, and the ISO finaled datetime became **{_iso}**. A naive `str[:10]` would have taken the first ten characters of `09/10/2020` (`09/10/2020`) and *not* a real ISO date at all — so every issuance date would have been left as un-parseable text and dropped.

The lesson is small but it repeats everywhere: **don't assume one format; ask a real parser to read the value.** One robust `pdate` now feeds every milestone below.''')

## Which date counts? Milestone selection, the real subtlety

A single building can carry many permit rows — an original permit plus a string of `-REV` and `-DEF` revisions. So which date is *the* building-permit date, and which is *the* completion date? Picking wrong here quietly corrupts every downstream year and RHNA-cycle count.

**Our plan:** for each building, reduce its **master** permits (revisions excluded) to exactly one date per milestone —
- **BP issued = MIN** over the masters: the *first* permit is when the building started, and a later revision must never reset that clock (this is what the RHNA cycle is earned on).
- **CO = the `Finaled` date** (MAX over masters): the building is done when its last real permit finals. This comes from the structured `Finaled Status` / `Finaled Date` columns — a genuine "occupiable" signal — **not** parsed from prose, so the event is honest (`is_inferred = 0`).

In [ ]:
# Collect, per building, the dates from its MASTER housing-creating permits (REV/DEF excluded).
bld = defaultdict(lambda: {'units': 0.0, 'hasnew': False, 'issue': [], 'final': []})
for r in df.itertuples(index=False):
    st = r.StreetType; st = '' if (st is None or str(st).strip().lower() == 'nan') else str(st)
    k = normalize_address(f'{r.StreetNumber} {r.StreetName} {st}'.strip())
    if not k.number: continue
    b = bld[(k.number, k.street, k.stype)]
    b['units'] = max(b['units'], net_units(r.isnew, r.UnitsAdded, r.NumberUnits, r.ADU))
    if r.isnew: b['hasnew'] = True
    pn = str(r.PermitNumber)
    if extract_master_permit(pn) == pn and net_units(r.isnew, r.UnitsAdded, r.NumberUnits, r.ADU) > 0:
        i, f = pdate(r.IssuanceDate), pdate(r.FinaledDate)
        if i: b['issue'].append((i, pn))
        if f: b['final'].append((f, pn))
spine = {k: b for k, b in bld.items() if b['hasnew'] or b['units'] > 0}

# emit one dated event per (building, milestone):
#   building_permit_issued = MIN over the building's master permits (the FIRST permit starts the clock)
#   co_issued              = MAX finaled over them (the building is done when its last real permit finals)
events = []
for k, b in spine.items():
    if b['issue']: events.append((k, 'building_permit_issued', min(b['issue'])[0]))
    if b['final']: events.append((k, 'co_issued',              max(b['final'])[0]))
n_bp = sum(1 for e in events if e[1] == 'building_permit_issued')
n_co = sum(1 for e in events if e[1] == 'co_issued')
print(f'{len(spine)} buildings -> {n_bp} BP-issued events + {n_co} co_issued events')


In [ ]:
md(f'''## What just happened

The {len(df):,} permit rows folded into **{len(spine)}** distinct buildings, and from their master permits we emitted two kinds of dated event: **{n_bp}** `building_permit_issued` events (one per building that has a BP, dated to its *earliest* master permit) and **{n_co}** `co_issued` events (dated to its *latest* finaled permit).

Read those two numbers against each other: there are **{n_bp - n_co}** more BP events than CO events ({n_bp} − {n_co}) — a rough sign of how many buildings have started but aren't finaled yet. Those two event streams are exactly what the next step turns into a *stage*. And every one of these events is a real date drawn from a structured column, never a guess.''')

## THE CORE LESSON: derive the stage, don't assert it

Here's the question this whole notebook is built around: how should a building know it's *completed*? The seductive shortcut is to read a column that already says `status = 'Completed'` and trust it. But a status string is just somebody's claim — it can be set, copied, or migrated wrong, with no date behind it.

We refuse that shortcut. A building's **stage** is a *conclusion we compute* from its dated events: **completed** if it has a real CO/finaled date, **permitted** if it has only a BP, **pipeline** if neither. (The migration once did it the lazy way — setting stage from a v1 status string — which is exactly how **14** buildings ended up marked `completed` with no event to back the claim.)

**Our plan:** define `stage_of` purely from the dated events, then count how many buildings fall into each stage.

In [ ]:
# STAGE is DERIVED from the dated events - never asserted from a status string.
def stage_of(b):
    if b['final']: return 'completed'    # has a real CO/finaled date
    if b['issue']: return 'permitted'    # a BP issued, but not yet finaled
    return 'pipeline'                    # neither - entitled/in-progress only
from collections import Counter
dist = Counter(stage_of(b) for b in spine.values())
print('stage distribution:', dict(dist), '  (the pipeline S3 = completed 951 / permitted 340 / pipeline 94)')


In [ ]:
md(f'''## What just happened

Every building sorted itself into exactly one stage **purely from its dates**: **{dist['completed']:,} completed** (they have a finaled date), **{dist['permitted']:,} permitted** (a BP but no finaled date yet), and **{dist['pipeline']:,} pipeline** (neither). Total: **{sum(dist.values()):,}** buildings.

The point isn't the three numbers — it's *where they came from*. Not one of them was read off a `status` string; each is a conclusion `stage_of` drew from events with real dates behind them. That is the difference between a number you can defend and the 14-building migration bug, where `completed` was asserted with nothing to prove it.''')

In [ ]:
import matplotlib.pyplot as plt
# draw the DERIVED stage distribution - the chart reads the computed dist, it does not recompute it
_order  = ['completed', 'permitted', 'pipeline']
_colors = {'completed': '#2e7d32', 'permitted': '#f39c12', 'pipeline': '#90a4ae'}
_vals   = [dist.get(s, 0) for s in _order]
fig, ax = plt.subplots(figsize=(7, 3.4))
bars = ax.bar(_order, _vals, color=[_colors[s] for s in _order])
for b, v in zip(bars, _vals):                       # label each bar with its count
    ax.text(b.get_x() + b.get_width() / 2, v, f'{v:,}', ha='center', va='bottom')
ax.set_title('Building stage - DERIVED from dated events, not asserted from a status string')
ax.set_ylabel('buildings'); ax.set_ylim(0, max(_vals) * 1.15)
plt.tight_layout(); plt.show()

### Does each stage hold up on a real building?

A distribution is convincing; three real addresses are *persuasive*. Can we point at one building in each stage and watch the same rule produce its label from its actual dates?

**Our plan:** print one worked example per stage — a completed building, a permitted one, and one still in the pipeline — and confirm that every completed building's CO is a structured date, with zero inferred.

In [ ]:
def show(name, addr):
    k = normalize_address(addr); b = spine[(k.number, k.street, k.stype)]
    bp = min(b['issue'])[0] if b['issue'] else None
    co = max(b['final'])[0] if b['final'] else None
    print(f'  {name:28} units={int(b["units"]):>4}  BP={bp}  CO={co}  -> {stage_of(b)}')
show('2001 Fourth (completed)', '2001 Fourth St')
show('1598 University (permitted)', '1598 University Ave')
show('2711 Shattuck (pipeline)', '2711 Shattuck Ave')

co_dates = [max(b['final'])[0] for b in spine.values() if b['final']]
print(f'\n  {len(co_dates)} completions, all with a structured CO date (0 inferred):', all(co_dates))

In [ ]:
def _stage(addr):
    k = normalize_address(addr); return stage_of(spine[(k.number, k.street, k.stype)])
_s_fourth = _stage('2001 Fourth St')
_s_univ   = _stage('1598 University Ave')
_s_shat   = _stage('2711 Shattuck Ave')
md(f'''## What just happened

Three real buildings, three different stages, all from the same rule reading their own dates: **2001 Fourth** has both a BP and a CO → **{_s_fourth}**; **1598 University** has a BP but no finaled date → **{_s_univ}**; **2711 Shattuck** has neither → **{_s_shat}**.

And the honesty check at the bottom held: all **{len(co_dates)}** completions carry a structured CO date, **0 inferred**. Every "completed" label in this notebook can be made to point at the date that earns it — which is precisely the claim a status string could never back up.''')

## (warning) Honesty note - the 2352 Shattuck thread, now in the DATE dimension

JN3 collapsed Logan Park to **one** 135-unit building. That collapse now bites the *date*: the building's CO is the **MAX finaled = 2023-08-08** (the South building's), which **mis-dates the North's 135 units** - they were finaled **2022-01-14**. Same collapse, new symptom. Still not fixed here; you will resolve it in JN6.

In [ ]:
k = normalize_address('2352 Shattuck Ave'); b = spine[(k.number, k.street, k.stype)]
print(f'2352 Shattuck: units={b["units"]:.0f}  CO (MAX-finaled) = {max(b["final"])[0]}')
print('  -> 135 North units carry the South building 2023 date. True North CO = 2022-01-14.')
print('  -> the address-collapse, surfacing in the date dimension. Rediscovered/resolved in JN6.')

## The checkpoint: verify before you trust

We've derived a stage for every building — but how do we *know* the derivation holds? Three cheap assertions pin the contract down: the completion count matches what the pipeline already knows is true (S3 = 951), every completion has a real CO date (0 inferred), and our three worked examples each land in the right stage. If someone later "simplifies" `stage_of` or lets a status string sneak back in, one of these breaks loudly — here, not three notebooks downstream.

In [ ]:
# 1) completions count matches the pipeline (S3 = 951)
completed = [k for k, b in spine.items() if b['final']]
assert len(completed) == 951, f'completed {len(completed)} != 951'
# 2) every completion has a real CO date (0 inferred / 0 guessed)
assert all(max(spine[k]['final'])[0] for k in completed)
# 3) one worked example of each stage, derived from events
def st(addr):
    k = normalize_address(addr); return stage_of(spine[(k.number, k.street, k.stype)])
assert st('2001 Fourth St')    == 'completed'
assert st('1598 University Ave') == 'permitted'
assert st('2711 Shattuck Ave')  == 'pipeline'

print('CHECKPOINT PASS')
print(f'  {len(completed)} completions (== S1/S3), all with a structured CO date (0 inferred)')
print('  stage DERIVED from events: 2001 Fourth=completed - 1598 University=permitted - 2711 Shattuck=pipeline')

In [ ]:
md(f'''## What just happened

All three assertions passed. There are exactly **{len(completed):,}** completions — the same number the pipeline reports at S3 — every one of them carrying a structured CO date with **0 inferred**, and the three worked examples each derived to the stage we expected (2001 Fourth → completed, 1598 University → permitted, 2711 Shattuck → pipeline).

That's the whole habit of this course in one cell: **state what must be true, then make the computer prove it.** A green checkpoint you can re-run beats a status string you hope was set right. JN5 picks these dated completions up and tags each with its reporting year and RHNA cycle.''')

**JN4 done.** Every building now has dated milestones and a stage *derived* from them, with zero asserted completions. **Next - JN5:** tagging each completion with its reporting year and RHNA cycle (and the three date-concepts that must not be conflated).

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← Previous: [JN3 · Build the spine + units](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN3_spine_units.ipynb)  |  Next: [JN5 · Year & cycle tagging](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN5_year_cycle.ipynb) →